# M25 — Train Networks in PyTorch

**Objective:** implement a full modern training loop with autograd.

M24 assigned blame with reverse-mode gradients on

`x → hidden_preactivation → hidden_activation → logits → probabilities → loss`.

M25 asks **how a training loop uses those gradients**. The useful whole is a
CPU `nn.Module` plus

`zero_grad → forward → loss → backward → step`

with train/eval mode, no-grad inference, protected splits, and a checkpoint.

Systematic multi-cause diagnosis stays closed (M26). A running notebook is
not competence.


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a sign, a layout, a mode flag, a first mismatch,
or whether parameters move.

CPU is canonical. Do not download weights, do not require a GPU, and do not
treat a decreasing loss as a debugging catalogue. If a failure is a missing
`zero_grad` or a train-mode evaluation, stay there.

The repository does not prefill learner answers, ADR text, or competence.


In [ ]:
from pathlib import Path
import inspect
import sys
import tempfile

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M25" / "pytorch_training.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M25.pytorch_training import (
    CANONICAL_DEVICE,
    CHECKPOINT_KEYS,
    DEFAULT_BATCH_SIZE,
    DEFAULT_EPOCHS,
    DEFAULT_LEARNING_RATE,
    DEFAULT_SEED,
    LOOP_ORDER,
    MODE_DROPOUT_P,
    N_HIDDEN,
    PARAMETER_LAYOUT,
    TRAIN_HIDDEN,
    assert_protected_splits,
    autograd_parity_report,
    batch_size_report,
    canonical_training_step,
    checkpoint_roundtrip,
    compact_run_report,
    evaluate,
    forward_parity_report,
    gradient_reset_experiment,
    held_out_eval,
    make_classification_fixture,
    make_loader,
    mean_softmax_nll,
    parameter_ownership,
    save_checkpoint,
    snapshot_parameters,
    teaching_batch,
    teaching_module,
    train_mode_eval_experiment,
    train_model,
    training_step,
)
from missions.M24.backprop_core import (
    REFERENCE_LOGITS,
    TEACHING_TARGETS,
    reference_backward,
)

import torch

print("repository root:", ROOT)
print("torch", torch.__version__, "canonical device:", CANONICAL_DEVICE)
print("loop order:", LOOP_ORDER)
print("teaching hidden:", N_HIDDEN, "train hidden:", TRAIN_HIDDEN)
print("layout:", PARAMETER_LAYOUT)
print("checkpoint keys:", CHECKPOINT_KEYS)


## M24 boundary: keep the reverse-mode numbers

M25 may compare autograd to M24 only on the same teaching graph, same
weights, same batch, and same mean softmax-NLL. The mapping that must
stay explicit:

- M24 dense map is `X @ W + b` with `W` shaped `(in, out)`
- `nn.Linear` stores `weight` as `(out, in)`, so autograd `W` is `linear.weight.T`

Do not "fix" a mismatch by changing M24. Import `missions.M24.backprop_core`
as the trusted reference.


## Frozen teaching fixtures

Declare the useful whole **before** the first calculation.

| Fixture | Value |
| --- | --- |
| Teaching graph | M24 `REFERENCE_*`, hidden ReLU, `3→2→3` |
| Teaching targets | class `0` for both rows (mean softmax NLL) |
| Linear layout | compare `linear.weight.T` with M24 `W` |
| Loop | `zero_grad → forward → loss → backward → step` |
| Device / dtype | CPU, `float64` |
| Train fixture | 36 clustered rows, splits 24 / 6 / 6, seed `2501` |
| Train net | `3→8→3` (named width change so the loop is observable) |
| Optimizer | SGD; teaching momentum `0`, train momentum `0.9` |
| Eval | `model.eval()` + `torch.no_grad()`; never `step` |
| Held-out | disjoint; scored only after freeze |

Primary sources: `pytorch-basics`, `fastai-course`, and
`karpathy-micrograd` in `data/source_registry.json`.


## Predict before running — parameter ownership

Timestamp a prediction before `run-module`.

Load the M24 teaching weights into a `TwoLayerNet`. Predict:

- registered names `fc1.weight`, `fc1.bias`, `fc2.weight`, `fc2.bias`
- `fc1.weight` shape `(2, 3)` even though M24 `W1` is `(3, 2)`
- every parameter is a leaf with `requires_grad=True` on CPU `float64`


In [ ]:
model = teaching_module()
ownership = parameter_ownership(model)
for row in ownership:
    print(row)
assert ownership[0]["name"] == "fc1.weight"
assert ownership[0]["shape"] == (2, 3)
assert ownership[0]["requires_grad"] and ownership[0]["is_leaf"]
assert ownership[0]["device"] == "cpu"
print("nn.Module owns the parameters; Linear stores W as (out, in)")


### Ownership is registration, not a screenshot

`named_parameters()` is the inventory. Autograd will write `.grad` onto
these tensors. `optimizer.step` will change their values. Neither of
those facts is visible from architecture artwork alone.


## Predict before running — forward parity before training

Timestamp a prediction before `run-forward-parity`.

With teaching weights and `model.eval()`, predict the M24 logits
`[[0, 0, 0], [1, 1.5, -0.25]]` and matching hidden activations. If
forward disagrees, do not train.


In [ ]:
forward = forward_parity_report(model)
print(forward)
print("M24 REFERENCE_LOGITS", REFERENCE_LOGITS)
assert forward["agrees"]
print("forward parity holds on the teaching micro-case")


### Names are still the addresses

Hidden pre-activation, hidden activation, logits, and probabilities must
match before autograd is trusted. Dropout is off (`p=0`) on the teaching
module, so train versus eval does not yet change the forward.


## Predict before running — autograd versus M24

Timestamp a prediction before `run-autograd-parity`.

Invariant: same teaching batch and mean softmax-NLL. Change only the
engine (autograd instead of manual reverse mode). Predict selected
`W2[0,0]` matches M24 (`≈ -0.329655`) after transposing Linear grads.


In [ ]:
autograd = autograd_parity_report(model)
print("loss autograd", autograd["loss_autograd"], "loss M24", autograd["loss_m24"])
print("selected", autograd["selected_name"], autograd["selected_autograd"], autograd["selected_m24"])
for row in autograd["rows"]:
    print(row)
m24 = reference_backward()
print("M24 dW2[0,0]", float(m24.d_W2[0, 0]), "targets", TEACHING_TARGETS)
assert autograd["agrees"]
print("autograd matches M24 on W1, b1, W2, b2")


### Autograd produces `.grad`, not an update

Agreement here means reverse-mode numbers survived the framework wrapper.
It does not mean the optimizer ran. `.grad` can sit on a leaf while the
parameter value stays put — the next cells separate those objects.


## Predict before running — one canonical training step

Timestamp a prediction before `run-one-step`.

On the teaching batch, SGD with momentum `0` and learning rate `0.25`
runs `zero_grad → forward → loss → backward → step`. Predict loss after
the step is lower, and that **every** parameter with a nonzero grad
moves — unlike M24's one-entry update.


In [ ]:
print("LOOP_ORDER", LOOP_ORDER)
step = canonical_training_step()
print("loss before/after", step["loss_before"], step["loss_after"])
print(step["trace"])
assert step["loss_after"] < step["loss_before"]
assert step["parameters_moved"]
assert step["trace"].zero_grad_called
assert list(step["order"]) == list(LOOP_ORDER)
print("one SGD step on all parameters; not convergence")


### Local movement is not a training program

The teaching loss went down once. That supports the autograd sign under
SGD with momentum 0 (`parameter - lr * gradient`). It is not an epoch
loop, and it is not evidence that validation or held-out data were
handled correctly.


## Predict before running — protected splits

Timestamp a prediction before `run-splits`.

The fixture has 36 rows: 8 train / 2 val / 2 held-out per class.
Predict the three index sets are disjoint, cover `0..35`, and that
held-out membership is a property of the split — not of whoever scores
last.


In [ ]:
splits = make_classification_fixture()
assert assert_protected_splits(splits)
print("n", int(splits.features.shape[0]), "train", len(splits.train_idx), "val", len(splits.val_idx), "held-out", len(splits.held_out_idx))
print("train[:6]", splits.train_idx[:6])
print("val", splits.val_idx)
print("held-out", splits.held_out_idx)
train_x, train_y = splits.train()
print("train batch shape", tuple(train_x.shape), tuple(train_y.shape))
assert set(splits.train_idx).isdisjoint(splits.held_out_idx)
print("held-out cannot enter the optimizer if the loop respects these indices")


### Held-out is a door, not a leftover

Validation may run each epoch. Held-out waits until the accepted
checkpoint. Mixing those two is an evaluation defect even when the
architecture is right.


## Predict before running — batch-size control

Timestamp a prediction before `run-batch`.

Change **only** batch size (`4` versus `12`). Hold the fixture, epoch
budget, SGD family, and seed. Predict steps per epoch change
(`24/4 = 6` versus `24/12 = 2`). Do not predict that the loss curves
are the same.


In [ ]:
batch_rows = batch_size_report(batch_sizes=(4, 12), epochs=3)
for row in batch_rows:
    print(row)
assert batch_rows[0]["steps_per_epoch"] == 6
assert batch_rows[1]["steps_per_epoch"] == 2
print("batch size changed the number of updates; no invariance claim")


### Dynamics are not a theorem

Different batch sizes take different numbers of SGD steps per epoch.
Curves can move. Record the control; do not promote a coincidence into
a law.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for row, color in zip(batch_rows, ("#4c78a8", "#d62728")):
    ax.plot(range(len(row["train_losses"])), row["train_losses"], marker="o", color=color, label=f"batch {row['batch_size']}")
ax.set_xlabel("epoch")
ax.set_ylabel("mean train loss")
ax.set_title("Batch size changes update count (same data, seed, SGD family)")
ax.legend()
fig.tight_layout()
plt.show()
print("each point is mean training loss for that epoch; steps per epoch differ")


## Predict before running — instrumented training loop

Timestamp a prediction before `run-train-loop`.

Train `3→8→3` on the train split only, validate each epoch, leave
held-out untouched. Predict train loss falls across 8 epochs, and that
every recorded training step has `zero_grad_called=True`.


In [ ]:
run = train_model()
print(compact_run_report(run))
assert run.epoch_traces[-1].train_loss < run.epoch_traces[0].train_loss
assert all(step.zero_grad_called for epoch in run.epoch_traces for step in epoch.steps)
assert run.device == "cpu"
print("train hidden", run.n_hidden, "epochs", run.epochs, "batch", run.batch_size)
print("held-out indices were not passed to training_step")


### Traces are the handoff

Each epoch records train loss, val loss, step count, and whether
`zero_grad` ran. That is what M26 will later break one control at a
time. Decreasing train loss on a 36-row toy fixture is not a
generalization claim.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
epochs = [row.epoch for row in run.epoch_traces]
ax.plot(epochs, [row.train_loss for row in run.epoch_traces], marker="o", label="train")
ax.plot(epochs, [row.val_loss for row in run.epoch_traces], marker="s", label="val")
ax.set_xlabel("epoch")
ax.set_ylabel("mean softmax-NLL")
ax.set_title("Train vs val on the protected fixture (held-out still frozen)")
ax.legend()
fig.tight_layout()
plt.show()
print("val is allowed during training; held-out is not on this figure")


## Predict before running — eval / no-grad held-out

Timestamp a prediction before `run-eval`.

Score held-out with `evaluate` / `held_out_eval`. Predict
`model.training is False`, `grad_enabled is False`, and parameters do
not move. A metric is not a training step.


In [ ]:
before = snapshot_parameters(run.model)
held = held_out_eval(run)
val_x, val_y = run.splits.val()
val_trace = evaluate(run.model, val_x, val_y, split="val")
after = snapshot_parameters(run.model)
print(held)
print("val", val_trace.loss, val_trace.accuracy, "training flag", val_trace.model_training)
assert held.split == "held_out"
assert not held.parameters_updated
assert not held.model_training
assert not held.grad_enabled
assert before == after
print("evaluation did not update parameters")


### Two flags, one freeze

`model.eval()` changes Dropout/BatchNorm behavior. `torch.no_grad()`
skips graph construction. Neither updates parameters. Forgetting either
is a different defect; mixing them with `optimizer.step` is training.


## Predict before running — checkpoint replay

Timestamp a prediction before `run-checkpoint`.

Save the accepted run (weights, optimizer state, epoch, seeds, widths,
RNG). Reload into a fresh module. Predict held-out logits match, and
that optimizer state is a different object from `.grad`.


In [ ]:
ckpt_dir = Path(tempfile.mkdtemp())
ckpt_path = ckpt_dir / "m25_accepted.pt"
ckpt = checkpoint_roundtrip(run, ckpt_path)
print(ckpt)
print("saved payload keys include", CHECKPOINT_KEYS)
assert ckpt["held_out_agrees"]
assert ckpt["replay_agrees"]
assert not ckpt["parameters_updated"]
print("round-trip held-out inference matched; ADR still unfilled")


### Weights are not a run

Momentum buffers live in the optimizer. Seeds and split policy live in
the payload. The unfilled ADR is which of those V05 will require before
calling a checkpoint accepted.


## Code reading — register, reset, step, mode, checkpoint, held-out

Read `training_step`, `evaluate`, and `save_checkpoint`. Predict before
the next cell:

- what `.grad` holds after `backward` and before `step`
- why a second reverse pass without `zero_grad` doubles a teaching grad
- whether `evaluate` can change parameter tensors
- which checkpoint keys a resume needs that inference does not


In [ ]:
print(inspect.getsource(training_step))
print("---")
print(inspect.getsource(evaluate).split("if updated:")[0])
print("---")
print(inspect.getsource(save_checkpoint))
x, y = teaching_batch()
demo = teaching_module()
demo.train()
loss = mean_softmax_nll()(demo(x), y)
loss.backward()
print("grad exists before step", demo.fc2.weight.grad is not None)
before = snapshot_parameters(demo)
# no optimizer.step here
assert before == snapshot_parameters(demo)
print("backward wrote .grad and left parameter values unchanged")


## Predict before running — Controlled failure: stale gradients

Timestamp a prediction before `run-failure`.

Keep the teaching batch, SGD family, and learning rate. Skip
`zero_grad` (`defect="stale_grad"`). Predict two reverse passes without
a step **add**, and two full steps **diverge** from the correctly reset
twin.


In [ ]:
reset = gradient_reset_experiment()
broken_step = canonical_training_step(defect="stale_grad")
print(reset)
print("broken step zero_grad_called", broken_step["trace"].zero_grad_called)
assert not broken_step["trace"].zero_grad_called
assert reset["reset_keeps_second_equal_first"]
assert reset["stale_second_is_sum"]
assert reset["updates_diverge"]
print("missing zero_grad accumulates leftover .grad")


### Diagnose before repair

Symptom: the loop still emits a finite loss and parameters still move,
but the second-step update does not match the reset twin.

Plausible hypotheses include a missing `zero_grad`, a doubled learning
rate, a wrong reduction, or a different batch. The discriminator is
already on the table: two backwards **without a step** on the same
teaching batch. Reset keeps the second `.grad` equal to the first.
Skip reset and the second `.grad` is the sum. A global LR bug would not
double `.grad` before `step`.


## Predict before running — Controlled failure: train-mode evaluation

Timestamp a prediction before `run-mode-failure`.

Same teaching weights, Dropout `p=0.5`. Compare `defect="none"`
(`eval` + `no_grad`) with `defect="train_mode_eval"`. Predict logits
differ and **parameters do not move** in either path.


In [ ]:
mode = train_mode_eval_experiment()
drop_model = teaching_module(dropout_p=MODE_DROPOUT_P)
x, y = teaching_batch()
wrong_eval = evaluate(drop_model, x, y, split="val", defect="train_mode_eval")
right_eval = evaluate(drop_model, x, y, split="val", defect="none")
print(mode)
print("explicit wrong flag", wrong_eval.model_training, "right flag", right_eval.model_training)
assert mode["logits_differ"]
assert not mode["parameters_updated_correct"]
assert not mode["parameters_updated_wrong"]
assert wrong_eval.model_training
assert not right_eval.model_training
print("train-mode evaluation changed outputs without training")


### Two defects, two discriminators

Stale grad poisons **updates** (accumulated `.grad`). Train-mode
evaluation poisons **metrics** (Dropout on) without `optimizer.step`.
The traces already record `zero_grad_called` and `model.training`.
Do not repair either defect by widening the net or by opening M26.


## Predict before running — smallest repair

Timestamp a prediction before `run-failure-repair`.

Predict `defect="none"` restores `zero_grad` on the teaching step and
`eval`/`no_grad` on held-out scoring, with the same module class, loss,
and optimizer family. Do not change teaching weights.


In [ ]:
repaired_step = canonical_training_step(defect="none")
repaired_eval = evaluate(teaching_module(), *teaching_batch(), split="held_out", defect="none")
print("repaired zero_grad", repaired_step["trace"].zero_grad_called)
print("repaired eval training flag", repaired_eval.model_training, "updated", repaired_eval.parameters_updated)
assert repaired_step["trace"].zero_grad_called
assert not repaired_eval.model_training
assert not repaired_eval.parameters_updated
print("restored gradient reset and eval/no-grad on the same loop")


## Evidence contract

Submit, in your own log (not in this repository):

- timestamped **Predict before running** notes
- the parameter-ownership table and Linear transpose
- forward parity and autograd parity versus M24
- one canonical loop step, with backward-without-step as a contrast
- disjoint split index sets
- batch-size dynamics without an invariance claim
- eval/no-grad held-out freeze
- checkpoint keys, including optimizer state versus `.grad`
- stale-grad / train-mode-eval diagnosis (symptom, hypotheses,
  discriminator, repair)

See `missions/M25/evidence_contract.yaml`. Do not paste filled evidence
into the committed notebook.


## No-AI gate

Close this notebook and complete `missions/M25/no_ai_gate.md` from a blank
page without AI-generated code, calculations, prose, or diagrams.

**Status:** [UNFILLED BY LEARNER]


## Unfilled ADR

Use `missions/M25/adr_prompt.md` to choose a V05 **training-loop
reproducibility / checkpoint** policy: seeds, saved state, optimizer
state, validation evidence, restart expectations, and rollback. Do not
claim global optimality.

- **Status:** [UNFILLED BY LEARNER]
- **Date:** [UNFILLED BY LEARNER]
- **Owner:** [UNFILLED BY LEARNER]
- **Decision:** [UNFILLED BY LEARNER]

This notebook is not that ADR. Formal engineering review is required at M25.


## M24 → M25 → M26 handoff

M24 assigned blame with reverse mode. M25 wrapped those numbers in an
instrumented PyTorch loop: module ownership, autograd parity, reset,
train/eval, protected splits, and checkpoints.

M26 may break data, optimization, architecture, or evaluation controls
**only after** this loop is defended. M25 does not teach that catalogue.


## Mission summary prompt

In your own words, using only numbers from this lab:

1. Why is `fc1.weight` shaped `(2, 3)` when M24 `W1` is `(3, 2)`?
2. What does `backward` write, and what does `step` write?
3. Why did two backwards without `zero_grad` double `W2[0,0]`'s grad?
4. How can Dropout change an evaluation metric without training?
5. What must M26 receive that a training-loss screenshot cannot provide?

Leave the answers in your evidence log, not in this file.


In [ ]:
assert forward["agrees"]
assert autograd["agrees"]
assert step["loss_after"] < step["loss_before"]
assert assert_protected_splits(splits)
assert batch_rows[0]["steps_per_epoch"] != batch_rows[1]["steps_per_epoch"]
assert run.epoch_traces[-1].train_loss < run.epoch_traces[0].train_loss
assert held.split == "held-out" or held.split == "held_out"
assert not held.parameters_updated
assert ckpt["held_out_agrees"]
assert reset["updates_diverge"]
assert mode["logits_differ"]
assert repaired_step["trace"].zero_grad_called
assert not repaired_eval.model_training
print("M25 integrity checks passed")
